<img src="https://www.th-koeln.de/img/logo.svg" style="float:right;" width="200">

# 12th exercise: <font color="#C70039">Q-table learning on a deterministic graph</font>

* Course: AML  
* Lecturer: <a href="https://www.gernotheisenberg.de/">Gernot Heisenberg</a>
* Author of notebook: <a href="https://www.gernotheisenberg.de/">Gernot Heisenberg</a>
* Date: 03.09.2026

---

**GENERAL NOTE 1**: 
Please make sure you are reading the entire notebook, since it contains a lot of information on your tasks (e.g. regarding the set of certain parameters or a specific computational trick), and the written mark downs as well as comments contain a lot of information on how things work together as a whole. 

**GENERAL NOTE 2**: 
* Please, when commenting source code, just use English language only. 
* When describing an observation please use English language, too.
* This applies to all exercises throughout this course.

---------------------------------

### <font color="FFC300">LEARNING OBJECTIVES</font>:

After this exercise, you can represent a finite decision problem as states, actions, and rewards; implement the tabular Q-learning update; explain exploration versus exploitation; and derive a route from a learned Q-table.

### <font color="ce33ff">DESCRIPTION</font>:

An agent moves between nine locations. A positive entry in the adjacency matrix represents a permitted move. The agent receives a reward of **10** when it enters the destination location `L4`; all other permitted moves have reward **0**. Episodes therefore end when `L4` is reached.

This deliberately small and deterministic problem isolates the Q-learning mechanism before we use a Gymnasium environment in Exercise 13.

In [1]:
import numpy as np

# A fixed generator makes experiments reproducible.
rng = np.random.default_rng(1)

location_to_state = {f"L{index}": index - 1 for index in range(1, 10)}
state_to_location = {state: location for location, state in location_to_state.items()}
goal_location = "L4"
goal_state = location_to_state[goal_location]

# Rows are current states and columns are possible next states.
adjacency = np.array([
    [0, 1, 0, 0, 0, 0, 0, 0, 0],
    [1, 0, 1, 0, 1, 0, 0, 0, 0],
    [0, 1, 0, 0, 0, 1, 0, 0, 0],
    [0, 0, 0, 0, 0, 0, 1, 0, 0],
    [0, 1, 0, 0, 0, 0, 0, 1, 0],
    [0, 0, 1, 0, 0, 0, 0, 0, 0],
    [0, 0, 0, 1, 0, 0, 0, 1, 0],
    [0, 0, 0, 0, 1, 0, 1, 0, 1],
    [0, 0, 0, 0, 0, 0, 0, 1, 0],
], dtype=int)

rewards = np.zeros_like(adjacency, dtype=float)
rewards[:, goal_state] = 10.0
rewards[adjacency == 0] = np.nan

print("State mapping:", location_to_state)
print("Destination:", goal_location)

State mapping: {'L1': 0, 'L2': 1, 'L3': 2, 'L4': 3, 'L5': 4, 'L6': 5, 'L7': 6, 'L8': 7, 'L9': 8}
Destination: L4


## Hyperparameters and Q-table

`alpha` controls how strongly new experience changes a Q-value. `gamma` discounts future reward. Epsilon starts high to encourage exploration and decreases after each episode.

Run the next cell once before completing the two functions below.

In [2]:
num_episodes = 1_000
max_steps_per_episode = 20
alpha = 0.7
gamma = 0.95
epsilon = 1.0
epsilon_min = 0.05
epsilon_decay = 0.995

# The table is zero-initialised. Action selection must still use reachable actions only.
q_table = np.zeros_like(rewards, dtype=float)

## <font color="FFC300">Task 1</font> — action selection

Complete `choose_action`. With probability epsilon it must choose a random *reachable* action. Otherwise it must choose the reachable action with the highest Q-value. Do not select an action from a zero entry of `adjacency`.

Explain in a Markdown cell why choosing from all nine actions would be incorrect.

In [3]:
def choose_action(state, q_table, epsilon):
    reachable_actions = np.flatnonzero(adjacency[state])

    # START STUDENT CODE
    # Return a random reachable action during exploration.
    # Return the reachable action with the largest Q-value during exploitation.
    raise NotImplementedError("Complete Task 1 before training.")
    # END STUDENT CODE

## <font color="FFC300">Task 2</font> — Q-learning update

Complete the update according to

$$Q(s,a) \leftarrow Q(s,a) + \alpha [r + \gamma \max_{a'} Q(s',a') - Q(s,a)].$$

For a terminal transition, the future value is zero. This is important because no action follows once the goal has been reached.

In [4]:
def update_q_value(state, action, next_state, reward, terminal):
    # START STUDENT CODE
    # Compute the future value, temporal-difference error, and updated Q-value.
    raise NotImplementedError("Complete Task 2 before training.")
    # END STUDENT CODE

## Training

After completing Tasks 1 and 2, run this cell. Each episode begins in a randomly selected non-goal state. The printed success rate should converge to one because the graph is deterministic and every sampled start state can reach `L4`.

In [5]:
episode_lengths = []

for episode in range(num_episodes):
    state = int(rng.integers(len(location_to_state)))
    while state == goal_state:
        state = int(rng.integers(len(location_to_state)))

    for step in range(max_steps_per_episode):
        action = choose_action(state, q_table, epsilon)
        next_state = action
        terminal = next_state == goal_state
        reward = rewards[state, action]
        update_q_value(state, action, next_state, reward, terminal)
        state = next_state

        if terminal:
            episode_lengths.append(step + 1)
            break
    else:
        episode_lengths.append(np.nan)

    epsilon = max(epsilon_min, epsilon * epsilon_decay)

print(f"Completed episodes: {np.isfinite(episode_lengths).mean():.1%}")
print(f"Mean successful episode length: {np.nanmean(episode_lengths):.2f}")

NotImplementedError: Complete Task 1 before training.

## <font color="FFC300">Task 3</font> — derive and evaluate a policy

Complete the policy function and use it to retrieve a route from `L9` to `L4`. Then change **one** hyperparameter at a time and record the effect on convergence and the learned route. Test at least `alpha`, `gamma`, initial epsilon, and epsilon decay.

Finally, alter one connection in `adjacency`. State your prediction before training again and explain the observed policy.

In [ ]:
def greedy_route(start_location, destination_location):
    route = [start_location]
    current_state = location_to_state[start_location]
    destination_state = location_to_state[destination_location]

    for _ in range(len(location_to_state) * 2):
        if current_state == destination_state:
            return route

        reachable_actions = np.flatnonzero(adjacency[current_state])
        # START STUDENT CODE
        # Select the reachable action with the largest Q-value and append its location.
        raise NotImplementedError("Complete Task 3 after training.")
        # END STUDENT CODE

    raise RuntimeError("The greedy policy did not reach the destination.")

print("Learned Q-table:")
print(np.round(q_table, 2))

## <font color="FFC300">Task 4</font> — Reflection

In your own words, answer the following:

1. Why is this decision problem deterministic?
2. Which values encode the goal-directed policy in the Q-table?
3. Why must the set of available actions depend on the current state?
4. What behaviour would you expect if epsilon never decayed?